# 03 — Leakage-Safe Feature Engineering

## Objectives

This notebook converts the frozen daily output from Notebook 02 into a modeling-ready dataset.

It will:

- verify the daily preprocessing handoff;
- create labels for exactly the next calendar day;
- build strictly historical lag and rolling features;
- document every predictor in a feature manifest;
- remove only rows without the required seven-day history;
- save and verify the modeling dataset.

This notebook does not split the data, scale features, select a model, tune a threshold, or train a model.


## 1. Environment and Shared-Storage Setup

The project data is stored in the shared Google Drive folder. The notebook reads the frozen Notebook 02 outputs directly from Drive and saves the Notebook 03 handoffs there.

No repository clone is required inside Colab.


In [1]:
from pathlib import Path
from IPython.display import display

import hashlib
import json
import sys

import numpy as np
import pandas as pd

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

STORAGE_ROOT = Path("/content/drive/MyDrive/CSE437_air_quality_group_18")

PATHS = {
    "processed": STORAGE_ROOT / "processed",
}

PATHS["processed"].mkdir(parents=True, exist_ok=True)

SELECTED_CITIES = [
    "Dhaka",
    "Dinājpur",
    "Bherāmāra",
    "Bhola",
    "Cox’s Bāzār",
]

COMMON_START = pd.Timestamp("2022-08-05")
COMMON_END = pd.Timestamp("2025-11-23")

EXPECTED_DATES_PER_CITY = 1207
EXPECTED_INPUT_ROWS = len(SELECTED_CITIES) * EXPECTED_DATES_PER_CITY
EXPECTED_INPUT_COLUMNS = 24

POLLUTANTS = ["pm10", "pm25", "co", "no2", "so2", "o3"]

INPUT_FILE = PATHS["processed"] / "daily_air_quality.csv"
PREPROCESSING_SUMMARY_FILE = (
    PATHS["processed"] / "notebook_02_preprocessing_summary.json"
)

OUTPUT_FILE = PATHS["processed"] / "modeling_dataset.csv"
FEATURE_MANIFEST_FILE = (
    PATHS["processed"] / "notebook_03_feature_manifest.csv"
)
FEATURE_SUMMARY_FILE = (
    PATHS["processed"] / "notebook_03_feature_summary.json"
)

print("Storage root:", STORAGE_ROOT)
print("Daily input:", INPUT_FILE)
print("Modeling output:", OUTPUT_FILE)


Mounted at /content/drive
Storage root: /content/drive/MyDrive/CSE437_air_quality_group_18
Daily input: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/daily_air_quality.csv
Modeling output: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/modeling_dataset.csv


## 2. Load and Validate the Frozen Daily Dataset

Notebook 03 must use the validated Notebook 02 output without changing its cities, dates, daily aggregation, or cleaning rules.

The checks below confirm the expected schema, dimensions, city coverage, complete daily calendar, missingness, ordering, and absence of previously created targets or features.


In [2]:
def calculate_sha256(file_path):
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


if not INPUT_FILE.is_file():
    raise FileNotFoundError(
        "Notebook 02 daily output was not found: "
        f"{INPUT_FILE}"
    )

if not PREPROCESSING_SUMMARY_FILE.is_file():
    raise FileNotFoundError(
        "Notebook 02 preprocessing summary was not found: "
        f"{PREPROCESSING_SUMMARY_FILE}"
    )

with PREPROCESSING_SUMMARY_FILE.open("r", encoding="utf-8") as file:
    preprocessing_summary = json.load(file)

input_checksum = calculate_sha256(INPUT_FILE)

daily = pd.read_csv(
    INPUT_FILE,
    parse_dates=["date"],
)

expected_columns = [
    "city",
    "date",
    "observed_rows",
    "observed_hours",
    "daily_aqi",
    "aqi_valid_hours",
]

for pollutant in POLLUTANTS:
    expected_columns.extend([
        f"{pollutant}_mean",
        f"{pollutant}_max",
        f"{pollutant}_valid_hours",
    ])

assert daily.shape == (EXPECTED_INPUT_ROWS, EXPECTED_INPUT_COLUMNS)
assert list(daily.columns) == expected_columns
assert list(daily.columns) == preprocessing_summary["output_columns"]
assert preprocessing_summary["selected_cities"] == SELECTED_CITIES
assert pd.Timestamp(preprocessing_summary["common_start"]) == COMMON_START
assert pd.Timestamp(preprocessing_summary["common_end"]) == COMMON_END
assert int(
    preprocessing_summary["expected_dates_per_city"]
) == EXPECTED_DATES_PER_CITY
assert int(
    preprocessing_summary["calendar_rows_after_reindex"]
) == EXPECTED_INPUT_ROWS
assert preprocessing_summary["created_target_or_model_features"] is False

assert set(daily["city"]) == set(SELECTED_CITIES)
assert daily["date"].min() == COMMON_START
assert daily["date"].max() == COMMON_END
assert not daily.duplicated(["city", "date"]).any()
assert daily.isna().sum().sum() == 0

sorted_city_dates = (
    daily[["city", "date"]]
    .sort_values(["city", "date"])
    .reset_index(drop=True)
)

assert daily[["city", "date"]].equals(sorted_city_dates)

full_calendar = pd.date_range(
    COMMON_START,
    COMMON_END,
    freq="D",
)

for city in SELECTED_CITIES:
    city_dates = daily.loc[
        daily["city"].eq(city),
        "date",
    ].tolist()

    assert city_dates == list(full_calendar)

forbidden_existing_terms = [
    "next_day",
    "target",
    "label",
    "lag",
    "rolling",
    "prediction",
    "split",
]

assert not any(
    term in column.lower()
    for column in daily.columns
    for term in forbidden_existing_terms
)

city_coverage = (
    daily.groupby("city")
    .agg(
        rows=("date", "size"),
        unique_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reindex(SELECTED_CITIES)
    .reset_index()
)

display(city_coverage)

print(f"Input rows: {len(daily):,}")
print(f"Input columns: {daily.shape[1]}")
print("Input SHA-256:", input_checksum)
print("Frozen Notebook 02 handoff verified.")


,city,rows,unique_dates,first_date,last_date
0,Dhaka,1207,1207,2022-08-05,2025-11-23
1,Dinājpur,1207,1207,2022-08-05,2025-11-23
2,Bherāmāra,1207,1207,2022-08-05,2025-11-23
3,Bhola,1207,1207,2022-08-05,2025-11-23
4,Cox’s Bāzār,1207,1207,2022-08-05,2025-11-23


Input rows: 6,035
Input columns: 24
Input SHA-256: 460009e01b010695a256be268112aacf1ad32b1757f69ab199fd3a7611d16aca
Frozen Notebook 02 handoff verified.


## 3. Lock the Feature Design Before Inspecting the Labels

The feature windows are fixed before examining target performance.

Thirteen historical signals are used:

- daily AQI;
- daily mean and maximum for PM10, PM2.5, CO, NO2, SO2, and O3.

For each signal, the notebook creates values from 1, 2, and 7 days before the target, plus shifted 3-day and 7-day rolling means. This produces 65 predictors.

The coverage-count columns are retained in the Notebook 02 handoff for auditing, but they are excluded as predictors here because they are constant at 24 throughout the selected scope. City and dates are retained as identifiers, not predictors. CO2 and calendar-season features are not introduced.


In [3]:
SIGNAL_COLUMNS = ["daily_aqi"]

for pollutant in POLLUTANTS:
    SIGNAL_COLUMNS.extend([
        f"{pollutant}_mean",
        f"{pollutant}_max",
    ])

LAG_DAYS = [1, 2, 7]
ROLLING_WINDOWS = [3, 7]

COVERAGE_COLUMNS = [
    "observed_rows",
    "observed_hours",
    "aqi_valid_hours",
    *[f"{pollutant}_valid_hours" for pollutant in POLLUTANTS],
]

coverage_diagnostic = pd.DataFrame({
    "minimum": daily[COVERAGE_COLUMNS].min(),
    "maximum": daily[COVERAGE_COLUMNS].max(),
    "unique_values": daily[COVERAGE_COLUMNS].nunique(),
})

assert len(SIGNAL_COLUMNS) == 13
assert coverage_diagnostic["unique_values"].eq(1).all()
assert coverage_diagnostic["minimum"].eq(24).all()
assert coverage_diagnostic["maximum"].eq(24).all()

feature_records = []
feature_columns = []

for source_column in SIGNAL_COLUMNS:
    for lag in LAG_DAYS:
        feature_name = f"{source_column}_lag{lag}"
        feature_columns.append(feature_name)
        feature_records.append({
            "feature": feature_name,
            "source_column": source_column,
            "feature_type": "individual_lag",
            "nearest_day_before_target": lag,
            "farthest_day_before_target": lag,
            "definition": f"Value exactly {lag} day(s) before target",
        })

    for window in ROLLING_WINDOWS:
        feature_name = f"{source_column}_roll{window}_mean"
        feature_columns.append(feature_name)
        feature_records.append({
            "feature": feature_name,
            "source_column": source_column,
            "feature_type": "shifted_rolling_mean",
            "nearest_day_before_target": 1,
            "farthest_day_before_target": window,
            "definition": (
                f"Mean of target-minus-1 through target-minus-{window}"
            ),
        })

feature_manifest = pd.DataFrame(feature_records)

feature_type_counts = (
    feature_manifest["feature_type"]
    .value_counts()
    .rename_axis("feature_type")
    .reset_index(name="feature_count")
)

assert len(feature_columns) == 65
assert len(feature_columns) == len(set(feature_columns))
assert feature_manifest["feature"].tolist() == feature_columns

display(coverage_diagnostic)
display(feature_type_counts)
display(feature_manifest.head(10))

print(f"Historical source signals: {len(SIGNAL_COLUMNS)}")
print(f"Planned predictor columns: {len(feature_columns)}")


,minimum,maximum,unique_values
observed_rows,24,24,1
observed_hours,24,24,1
aqi_valid_hours,24,24,1
pm10_valid_hours,24,24,1
pm25_valid_hours,24,24,1
co_valid_hours,24,24,1
no2_valid_hours,24,24,1
so2_valid_hours,24,24,1
o3_valid_hours,24,24,1


,feature_type,feature_count
0,individual_lag,39
1,shifted_rolling_mean,26


,feature,source_column,feature_type,nearest_day_before_target,farthest_day_before_target,definition
0,daily_aqi_lag1,daily_aqi,individual_lag,1,1,Value exactly 1 day(s) before target
1,daily_aqi_lag2,daily_aqi,individual_lag,2,2,Value exactly 2 day(s) before target
2,daily_aqi_lag7,daily_aqi,individual_lag,7,7,Value exactly 7 day(s) before target
3,daily_aqi_roll3_mean,daily_aqi,shifted_rolling_mean,1,3,Mean of target-minus-1 through target-minus-3
4,daily_aqi_roll7_mean,daily_aqi,shifted_rolling_mean,1,7,Mean of target-minus-1 through target-minus-7
5,pm10_mean_lag1,pm10_mean,individual_lag,1,1,Value exactly 1 day(s) before target
6,pm10_mean_lag2,pm10_mean,individual_lag,2,2,Value exactly 2 day(s) before target
7,pm10_mean_lag7,pm10_mean,individual_lag,7,7,Value exactly 7 day(s) before target
8,pm10_mean_roll3_mean,pm10_mean,shifted_rolling_mean,1,3,Mean of target-minus-1 through target-minus-3
9,pm10_mean_roll7_mean,pm10_mean,shifted_rolling_mean,1,7,Mean of target-minus-1 through target-minus-7


Historical source signals: 13
Planned predictor columns: 65


## 4. Construct the Exact Next-Calendar-Day Targets

Each row is organized around a target date. Its feature date is the immediately preceding calendar day.

For example, a target dated 12 August uses 11 August as the feature date. The main label is 1 when the target day's daily AQI is greater than 150. The optional sensitivity label is 1 when it is greater than 100.

The actual target-day AQI is retained as an outcome for later severity and error analysis, but it is never included in the predictor list.


In [4]:
modeling = (
    daily[["city", "date", "daily_aqi"]]
    .rename(
        columns={
            "date": "target_date",
            "daily_aqi": "target_daily_aqi",
        }
    )
    .copy()
)

modeling["feature_date"] = (
    daily.groupby("city", sort=False)["date"]
    .shift(1)
)

modeling["next_day_unhealthy"] = (
    modeling["target_daily_aqi"]
    .gt(150)
    .astype("int8")
)

modeling["next_day_usg_or_worse"] = (
    modeling["target_daily_aqi"]
    .gt(100)
    .astype("int8")
)

alignable_rows = modeling["feature_date"].notna()

date_difference = (
    modeling.loc[alignable_rows, "target_date"]
    - modeling.loc[alignable_rows, "feature_date"]
)

assert date_difference.eq(pd.Timedelta(days=1)).all()
assert modeling["target_daily_aqi"].notna().all()
assert set(modeling["next_day_unhealthy"].unique()).issubset({0, 1})
assert set(modeling["next_day_usg_or_worse"].unique()).issubset({0, 1})

alignment_preview = (
    modeling.loc[alignable_rows, [
        "city",
        "feature_date",
        "target_date",
        "target_daily_aqi",
        "next_day_unhealthy",
        "next_day_usg_or_worse",
    ]]
    .groupby("city", sort=False)
    .head(2)
)

display(alignment_preview)

print("Every available target is exactly one calendar day after its feature date.")


,city,feature_date,target_date,target_daily_aqi,next_day_unhealthy,next_day_usg_or_worse
1,Bherāmāra,2022-08-05,2022-08-06,52.0,0,0
2,Bherāmāra,2022-08-06,2022-08-07,51.0,0,0
1208,Bhola,2022-08-05,2022-08-06,27.0,0,0
1209,Bhola,2022-08-06,2022-08-07,32.0,0,0
2415,Cox’s Bāzār,2022-08-05,2022-08-06,26.0,0,0
2416,Cox’s Bāzār,2022-08-06,2022-08-07,32.0,0,0
3622,Dhaka,2022-08-05,2022-08-06,54.0,0,0
3623,Dhaka,2022-08-06,2022-08-07,53.0,0,0
4829,Dinājpur,2022-08-05,2022-08-06,63.0,0,0
4830,Dinājpur,2022-08-06,2022-08-07,59.0,0,0


Every available target is exactly one calendar day after its feature date.


## 5. Create Historical Lag and Shifted Rolling Features

All operations are performed separately within each city.

Individual lag features use measurements from exactly 1, 2, or 7 days before the target. Rolling features first shift the signal by one day and then calculate the trailing mean. Therefore, a 3-day rolling feature uses only target-minus-1 through target-minus-3.


In [5]:
grouped_daily = daily.groupby("city", sort=False)

for source_column in SIGNAL_COLUMNS:
    for lag in LAG_DAYS:
        feature_name = f"{source_column}_lag{lag}"

        modeling[feature_name] = (
            grouped_daily[source_column]
            .shift(lag)
        )

    for window in ROLLING_WINDOWS:
        feature_name = f"{source_column}_roll{window}_mean"

        modeling[feature_name] = (
            grouped_daily[source_column]
            .transform(
                lambda values, window=window: (
                    values.shift(1)
                    .rolling(
                        window=window,
                        min_periods=window,
                    )
                    .mean()
                )
            )
        )

assert set(feature_columns).issubset(modeling.columns)

preview_columns = [
    "city",
    "feature_date",
    "target_date",
    "target_daily_aqi",
    "next_day_unhealthy",
    *feature_columns[:6],
]

display(modeling.loc[alignable_rows, preview_columns].head(10))

print("Lag and shifted rolling features created.")


,city,feature_date,target_date,target_daily_aqi,next_day_unhealthy,daily_aqi_lag1,daily_aqi_lag2,daily_aqi_lag7,daily_aqi_roll3_mean,daily_aqi_roll7_mean,pm10_mean_lag1
1,Bherāmāra,2022-08-05,2022-08-06,52.0,0,53.0,NaN,NaN,NaN,NaN,18.466667
2,Bherāmāra,2022-08-06,2022-08-07,51.0,0,52.0,53.0,NaN,NaN,NaN,18.358333
3,Bherāmāra,2022-08-07,2022-08-08,48.0,0,51.0,52.0,NaN,52.000000,NaN,16.862500
4,Bherāmāra,2022-08-08,2022-08-09,34.0,0,48.0,51.0,NaN,50.333333,NaN,11.512500
5,Bherāmāra,2022-08-09,2022-08-10,34.0,0,34.0,48.0,NaN,44.333333,NaN,8.900000
6,Bherāmāra,2022-08-10,2022-08-11,53.0,0,34.0,34.0,NaN,38.666667,NaN,8.254167
7,Bherāmāra,2022-08-11,2022-08-12,67.0,0,53.0,34.0,53.0,40.333333,46.428571,20.529167
8,Bherāmāra,2022-08-12,2022-08-13,59.0,0,67.0,53.0,52.0,51.333333,48.428571,23.691667
9,Bherāmāra,2022-08-13,2022-08-14,42.0,0,59.0,67.0,51.0,59.666667,49.428571,14.662500
10,Bherāmāra,2022-08-14,2022-08-15,52.0,0,42.0,59.0,48.0,56.000000,48.142857,10.366667


Lag and shifted rolling features created.


## 6. Keep Rows With the Complete Seven-Day History

The longest feature looks seven days into the past. Consequently, the first seven target dates for each city cannot contain all 65 predictors.

Those structurally incomplete rows are removed. No pollutant or AQI value is imputed.


In [6]:
identifier_columns = [
    "city",
    "feature_date",
    "target_date",
]

outcome_columns = [
    "target_daily_aqi",
    "next_day_unhealthy",
    "next_day_usg_or_worse",
]

required_columns = [
    *identifier_columns,
    *outcome_columns,
    *feature_columns,
]

complete_row_mask = (
    modeling[required_columns]
    .notna()
    .all(axis=1)
)

modeling_final = (
    modeling.loc[complete_row_mask, required_columns]
    .copy()
)

maximum_history_days = max([
    *LAG_DAYS,
    *ROLLING_WINDOWS,
])

expected_rows_per_city = (
    EXPECTED_DATES_PER_CITY
    - maximum_history_days
)

expected_output_rows = (
    len(SELECTED_CITIES)
    * expected_rows_per_city
)

input_rows_by_city = daily.groupby("city").size()
output_rows_by_city = modeling_final.groupby("city").size()

row_accounting_by_city = pd.DataFrame({
    "input_rows": input_rows_by_city,
    "modeling_rows": output_rows_by_city,
})

row_accounting_by_city["rows_removed"] = (
    row_accounting_by_city["input_rows"]
    - row_accounting_by_city["modeling_rows"]
)

row_accounting_by_city = (
    row_accounting_by_city
    .reindex(SELECTED_CITIES)
    .reset_index()
)

overall_row_accounting = pd.DataFrame({
    "stage": [
        "Frozen daily input",
        "Complete-history modeling output",
        "Rows removed for seven-day history",
    ],
    "rows": [
        len(daily),
        len(modeling_final),
        len(daily) - len(modeling_final),
    ],
})

assert len(modeling_final) == expected_output_rows
assert output_rows_by_city.eq(expected_rows_per_city).all()
assert row_accounting_by_city["rows_removed"].eq(
    maximum_history_days
).all()
assert modeling_final[feature_columns].notna().all().all()
assert modeling_final["target_date"].min() == (
    COMMON_START + pd.Timedelta(days=maximum_history_days)
)
assert modeling_final["target_date"].max() == COMMON_END
assert modeling_final["feature_date"].min() == (
    COMMON_START + pd.Timedelta(days=maximum_history_days - 1)
)
assert modeling_final["feature_date"].max() == (
    COMMON_END - pd.Timedelta(days=1)
)

display(overall_row_accounting)
display(row_accounting_by_city)
display(modeling_final.head())

print(f"Final modeling rows: {len(modeling_final):,}")
print(f"Rows per city: {expected_rows_per_city:,}")


,stage,rows
0,Frozen daily input,6035
1,Complete-history modeling output,6000
2,Rows removed for seven-day history,35


,city,input_rows,modeling_rows,rows_removed
0,Dhaka,1207,1200,7
1,Dinājpur,1207,1200,7
2,Bherāmāra,1207,1200,7
3,Bhola,1207,1200,7
4,Cox’s Bāzār,1207,1200,7


,city,feature_date,target_date,target_daily_aqi,next_day_unhealthy,next_day_usg_or_worse,daily_aqi_lag1,daily_aqi_lag2,daily_aqi_lag7,daily_aqi_roll3_mean,...,o3_mean_lag1,o3_mean_lag2,o3_mean_lag7,o3_mean_roll3_mean,o3_mean_roll7_mean,o3_max_lag1,o3_max_lag2,o3_max_lag7,o3_max_roll3_mean,o3_max_roll7_mean
7,Bherāmāra,2022-08-11,2022-08-12,67.0,0,0,53.0,34.0,53.0,40.333333,...,31.958333,58.625000,36.958333,49.583333,45.785714,56.0,82.0,84.0,72.666667,80.857143
8,Bherāmāra,2022-08-12,2022-08-13,59.0,0,0,67.0,53.0,52.0,51.333333,...,50.041667,31.958333,30.916667,46.875000,47.654762,109.0,56.0,72.0,82.333333,84.428571
9,Bherāmāra,2022-08-13,2022-08-14,42.0,0,0,59.0,67.0,51.0,59.666667,...,56.750000,50.041667,49.958333,46.250000,51.345238,96.0,109.0,106.0,87.000000,87.857143
10,Bherāmāra,2022-08-14,2022-08-15,52.0,0,0,42.0,59.0,48.0,56.000000,...,48.958333,56.750000,53.916667,51.916667,51.202381,67.0,96.0,86.0,90.666667,82.285714
11,Bherāmāra,2022-08-15,2022-08-16,96.0,0,0,52.0,42.0,34.0,51.000000,...,44.208333,48.958333,58.166667,49.972222,49.815476,81.0,67.0,80.0,81.333333,81.571429


Final modeling rows: 6,000
Rows per city: 1,200


## 7. Inspect Feature Completeness and Target Balance

These diagnostics describe the finished modeling handoff. They do not select feature windows, create data splits, or compare models.

Target prevalence is reported separately for every city because accuracy can be misleading when class proportions differ.


In [7]:
feature_missingness = pd.DataFrame({
    "missing_count": modeling_final[feature_columns].isna().sum(),
    "missing_percent": (
        modeling_final[feature_columns]
        .isna()
        .mean()
        .mul(100)
    ),
})

target_balance = (
    modeling_final.groupby("city")
    .agg(
        modeling_rows=("target_date", "size"),
        unhealthy_days=("next_day_unhealthy", "sum"),
        unhealthy_rate=("next_day_unhealthy", "mean"),
        usg_or_worse_days=("next_day_usg_or_worse", "sum"),
        usg_or_worse_rate=("next_day_usg_or_worse", "mean"),
    )
    .reindex(SELECTED_CITIES)
    .reset_index()
)

target_balance[[
    "unhealthy_rate",
    "usg_or_worse_rate",
]] = (
    target_balance[[
        "unhealthy_rate",
        "usg_or_worse_rate",
    ]]
    .mul(100)
    .round(2)
)

overall_target_balance = pd.DataFrame({
    "target": [
        "Next-day unhealthy: AQI > 150",
        "Next-day USG-or-worse: AQI > 100",
    ],
    "positive_rows": [
        int(modeling_final["next_day_unhealthy"].sum()),
        int(modeling_final["next_day_usg_or_worse"].sum()),
    ],
    "positive_rate_percent": [
        round(
            modeling_final["next_day_unhealthy"].mean() * 100,
            2,
        ),
        round(
            modeling_final["next_day_usg_or_worse"].mean() * 100,
            2,
        ),
    ],
})

assert feature_missingness["missing_count"].eq(0).all()

display(feature_type_counts)
display(feature_missingness)
display(target_balance)
display(overall_target_balance)


,feature_type,feature_count
0,individual_lag,39
1,shifted_rolling_mean,26


,missing_count,missing_percent
daily_aqi_lag1,0,0.0
daily_aqi_lag2,0,0.0
daily_aqi_lag7,0,0.0
daily_aqi_roll3_mean,0,0.0
daily_aqi_roll7_mean,0,0.0
...,...,...
o3_max_lag1,0,0.0
o3_max_lag2,0,0.0
o3_max_lag7,0,0.0
o3_max_roll3_mean,0,0.0


,city,modeling_rows,unhealthy_days,unhealthy_rate,usg_or_worse_days,usg_or_worse_rate
0,Dhaka,1200,563,46.92,788,65.67
1,Dinājpur,1200,540,45.00,889,74.08
2,Bherāmāra,1200,660,55.00,885,73.75
3,Bhola,1200,339,28.25,579,48.25
4,Cox’s Bāzār,1200,297,24.75,562,46.83


,target,positive_rows,positive_rate_percent
0,Next-day unhealthy: AQI > 150,2399,39.98
1,Next-day USG-or-worse: AQI > 100,3703,61.72


## 8. Leakage and Integrity Assertions

The gate below verifies that:

- every target is exactly one day after the feature date;
- every documented predictor begins at least one day before the target;
- lag-1 values match the measurements recorded on the feature date;
- rolling features were shifted before rolling;
- target and identifier columns are excluded from the predictor list;
- all 65 predictors are finite numeric values;
- the output contains exactly one row per city-target-date.

Notebook 04 must use the saved feature list and must not treat the identifiers or outcome columns as predictors.


In [8]:
assert not modeling_final.duplicated([
    "city",
    "target_date",
]).any()

sorted_output_dates = (
    modeling_final[["city", "target_date"]]
    .sort_values(["city", "target_date"])
    .reset_index(drop=True)
)

assert (
    modeling_final[["city", "target_date"]]
    .reset_index(drop=True)
    .equals(sorted_output_dates)
)

assert (
    modeling_final["target_date"]
    - modeling_final["feature_date"]
).eq(pd.Timedelta(days=1)).all()

assert feature_manifest[
    "nearest_day_before_target"
].ge(1).all()

assert feature_manifest[
    "farthest_day_before_target"
].ge(
    feature_manifest["nearest_day_before_target"]
).all()

forbidden_feature_terms = [
    "target",
    "next_day",
    "feature_date",
    "target_date",
    "city",
]

assert not any(
    term in feature.lower()
    for feature in feature_columns
    for term in forbidden_feature_terms
)

assert set(identifier_columns).isdisjoint(feature_columns)
assert set(outcome_columns).isdisjoint(feature_columns)
assert modeling_final[feature_columns].apply(
    pd.api.types.is_numeric_dtype
).all()
assert np.isfinite(
    modeling_final[feature_columns].to_numpy(dtype=float)
).all()

expected_main_target = (
    modeling_final["target_daily_aqi"]
    .gt(150)
    .astype("int8")
)

expected_sensitivity_target = (
    modeling_final["target_daily_aqi"]
    .gt(100)
    .astype("int8")
)

assert modeling_final["next_day_unhealthy"].equals(
    expected_main_target
)
assert modeling_final["next_day_usg_or_worse"].equals(
    expected_sensitivity_target
)

source_by_city_date = daily.set_index(["city", "date"])

for source_column in SIGNAL_COLUMNS:
    for lag in LAG_DAYS:
        source_dates = (
            modeling_final["target_date"]
            - pd.to_timedelta(lag, unit="D")
        )

        source_index = pd.MultiIndex.from_arrays(
            [
                modeling_final["city"],
                source_dates,
            ],
            names=["city", "date"],
        )

        expected_values = (
            source_by_city_date[source_column]
            .reindex(source_index)
            .to_numpy()
        )

        actual_values = modeling_final[
            f"{source_column}_lag{lag}"
        ].to_numpy()

        assert np.allclose(
            actual_values,
            expected_values,
            equal_nan=True,
        )

for source_column in SIGNAL_COLUMNS:
    for window in ROLLING_WINDOWS:
        expected_values = (
            grouped_daily[source_column]
            .transform(
                lambda values, window=window: (
                    values.shift(1)
                    .rolling(
                        window=window,
                        min_periods=window,
                    )
                    .mean()
                )
            )
            .loc[modeling_final.index]
            .to_numpy()
        )

        actual_values = modeling_final[
            f"{source_column}_roll{window}_mean"
        ].to_numpy()

        assert np.allclose(
            actual_values,
            expected_values,
            equal_nan=True,
        )

assert len(feature_columns) == 65
assert len(modeling_final.columns) == (
    len(identifier_columns)
    + len(outcome_columns)
    + len(feature_columns)
)

print("All Notebook 03 leakage and integrity checks passed.")


All Notebook 03 leakage and integrity checks passed.


## 9. Save and Verify the Notebook 03 Handoffs

The modeling dataset, feature manifest, and compact JSON summary are saved in shared storage.

The CSV is read back after saving so its dimensions, column order, dates, labels, uniqueness, and feature completeness can be verified.


In [9]:
modeling_to_save = modeling_final.copy()

for date_column in ["feature_date", "target_date"]:
    modeling_to_save[date_column] = (
        modeling_to_save[date_column]
        .dt.strftime("%Y-%m-%d")
    )

modeling_to_save.to_csv(
    OUTPUT_FILE,
    index=False,
)

feature_manifest.to_csv(
    FEATURE_MANIFEST_FILE,
    index=False,
)

feature_summary = {
    "input_file": INPUT_FILE.name,
    "input_sha256": input_checksum,
    "input_rows": int(len(daily)),
    "input_columns": int(daily.shape[1]),
    "selected_cities": SELECTED_CITIES,
    "common_start": str(COMMON_START.date()),
    "common_end": str(COMMON_END.date()),
    "target_definition": (
        "Next calendar day's daily maximum AQI > 150"
    ),
    "sensitivity_target_definition": (
        "Next calendar day's daily maximum AQI > 100"
    ),
    "target_alignment": (
        "target_date is exactly feature_date plus one calendar day"
    ),
    "signal_columns": SIGNAL_COLUMNS,
    "lag_days_before_target": LAG_DAYS,
    "rolling_windows_days": ROLLING_WINDOWS,
    "rolling_rule": (
        "Shift by one day before every rolling calculation"
    ),
    "coverage_predictor_decision": (
        "Excluded because all coverage-count columns are constant at 24"
    ),
    "calendar_feature_decision": (
        "No month or season predictors; seasonality remains descriptive only"
    ),
    "output_rows": int(len(modeling_final)),
    "output_rows_per_city": int(expected_rows_per_city),
    "rows_removed_for_history": int(
        len(daily) - len(modeling_final)
    ),
    "feature_count": int(len(feature_columns)),
    "identifier_columns": identifier_columns,
    "outcome_columns": outcome_columns,
    "feature_columns": feature_columns,
    "created_split_scaler_or_model": False,
    "row_accounting_by_city": json.loads(
        row_accounting_by_city.to_json(
            orient="records",
            date_format="iso",
        )
    ),
    "target_balance_by_city": json.loads(
        target_balance.to_json(
            orient="records",
            date_format="iso",
        )
    ),
    "output_file": OUTPUT_FILE.name,
    "feature_manifest_file": FEATURE_MANIFEST_FILE.name,
}

with FEATURE_SUMMARY_FILE.open("w", encoding="utf-8") as file:
    json.dump(
        feature_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )

saved_modeling = pd.read_csv(
    OUTPUT_FILE,
    parse_dates=["feature_date", "target_date"],
)

saved_manifest = pd.read_csv(
    FEATURE_MANIFEST_FILE,
)

assert saved_modeling.shape == modeling_final.shape
assert list(saved_modeling.columns) == list(modeling_final.columns)
assert not saved_modeling.duplicated([
    "city",
    "target_date",
]).any()
assert (
    saved_modeling["target_date"]
    - saved_modeling["feature_date"]
).eq(pd.Timedelta(days=1)).all()
assert saved_modeling[feature_columns].notna().all().all()
assert set(saved_modeling["next_day_unhealthy"].unique()).issubset(
    {0, 1}
)
assert set(
    saved_modeling["next_day_usg_or_worse"].unique()
).issubset({0, 1})
assert saved_manifest["feature"].tolist() == feature_columns
assert calculate_sha256(INPUT_FILE) == input_checksum

print("Saved:", OUTPUT_FILE)
print("Saved:", FEATURE_MANIFEST_FILE)
print("Saved:", FEATURE_SUMMARY_FILE)
print(f"Verified saved rows: {len(saved_modeling):,}")
print(f"Verified saved columns: {saved_modeling.shape[1]}")
print(f"Verified predictor columns: {len(feature_columns)}")
print("\nNOTEBOOK 03 FEATURE ENGINEERING GATE PASSED")

display(saved_modeling.head())


Saved: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/modeling_dataset.csv
Saved: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/notebook_03_feature_manifest.csv
Saved: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/notebook_03_feature_summary.json
Verified saved rows: 6,000
Verified saved columns: 71
Verified predictor columns: 65

NOTEBOOK 03 FEATURE ENGINEERING GATE PASSED


,city,feature_date,target_date,target_daily_aqi,next_day_unhealthy,next_day_usg_or_worse,daily_aqi_lag1,daily_aqi_lag2,daily_aqi_lag7,daily_aqi_roll3_mean,...,o3_mean_lag1,o3_mean_lag2,o3_mean_lag7,o3_mean_roll3_mean,o3_mean_roll7_mean,o3_max_lag1,o3_max_lag2,o3_max_lag7,o3_max_roll3_mean,o3_max_roll7_mean
0,Bherāmāra,2022-08-11,2022-08-12,67.0,0,0,53.0,34.0,53.0,40.333333,...,31.958333,58.625000,36.958333,49.583333,45.785714,56.0,82.0,84.0,72.666667,80.857143
1,Bherāmāra,2022-08-12,2022-08-13,59.0,0,0,67.0,53.0,52.0,51.333333,...,50.041667,31.958333,30.916667,46.875000,47.654762,109.0,56.0,72.0,82.333333,84.428571
2,Bherāmāra,2022-08-13,2022-08-14,42.0,0,0,59.0,67.0,51.0,59.666667,...,56.750000,50.041667,49.958333,46.250000,51.345238,96.0,109.0,106.0,87.000000,87.857143
3,Bherāmāra,2022-08-14,2022-08-15,52.0,0,0,42.0,59.0,48.0,56.000000,...,48.958333,56.750000,53.916667,51.916667,51.202381,67.0,96.0,86.0,90.666667,82.285714
4,Bherāmāra,2022-08-15,2022-08-16,96.0,0,0,52.0,42.0,34.0,51.000000,...,44.208333,48.958333,58.166667,49.972222,49.815476,81.0,67.0,80.0,81.333333,81.571429


## 10. Notebook 03 Handoff

If the previous cell prints NOTEBOOK 03 FEATURE ENGINEERING GATE PASSED, the leakage-safe modeling dataset is ready for review.

Notebook 04 may then:

1. load the frozen modeling dataset and feature manifest;
2. define chronological train, validation, and untouched test periods;
3. build the persistence baseline before any machine-learning model;
4. fit preprocessing steps using training data only;
5. compare a small, justified set of models using validation data;
6. run the Dhaka-to-smaller-city transfer experiment.

Notebook 03 deliberately contains no random split, chronological split, scaling, feature selection based on target performance, threshold tuning, prediction, or model training.
